# D-SAT — Phase 1: YouCook2 Data Pipeline

**Objective:** Produce a structured dataset of `(G_t, action_text, G_t+1)` causal triplets from YouCook2 cooking videos.

Each triplet contains:
- `G_t` — a JSON scene graph representing the *start* frame of a captioned action step
- `action_text` — the human-annotated caption (e.g. `"pour the dressing over the salad"`)
- `G_t+1` — a JSON scene graph representing the *end* frame of that same step

**Pipeline stages:**
```
1. Environment setup
2. Load YouCook2 annotations (HuggingFace)
3. Download video segments with yt-dlp
4. Extract frame pairs (start frame, end frame)
5. Call Teacher VLM (Gemini 2.5 Flash) to generate G_t and G_t+1
6. Consistency filtering
7. Save final triplet dataset as JSONL
```

> **Runtime:** `T4 GPU` is sufficient for this phase (no model training).
> Add your Gemini API key to Colab Secrets as `GEMINI_API_KEY` before running.

---
## Stage 1 — Install dependencies

In [ ]:
%%capture
!pip install yt-dlp datasets google-generativeai Pillow tqdm sentence-transformers
!apt-get install -y ffmpeg

In [ ]:
import os, json, time, textwrap
from pathlib import Path
from typing import Optional
from collections import defaultdict

import cv2
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from google import genai
from google.genai import types
from datasets import load_dataset

# API key — loaded from Colab Secrets if available, otherwise prompted
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    import getpass
    GEMINI_API_KEY = getpass.getpass("Gemini API key: ")

# Directory structure
ROOT        = Path("/content/dsat")
VIDEO_DIR   = ROOT / "videos"
FRAMES_DIR  = ROOT / "frames"
GRAPHS_DIR  = ROOT / "graphs"
OUTPUT_FILE = ROOT / "triplets.jsonl"

for d in [VIDEO_DIR, FRAMES_DIR, GRAPHS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Gemini client
client = genai.Client(api_key=GEMINI_API_KEY)
TEACHER_MODEL_ID = "models/gemini-2.5-flash-lite"

print("Setup complete. Teacher model:", TEACHER_MODEL_ID)

---
## Stage 2 — Load YouCook2 annotations

Using the HuggingFace mirror (`lmms-lab/YouCook2`). Each row contains: `youtube_id`, `segment` ([t_start, t_end] in seconds), `sentence`, and `recipe_type`.

The `val` split (3,179 segments across 414 videos) is used first to validate the pipeline before scaling to the full training set.

In [ ]:
ds = load_dataset("lmms-lab/YouCook2", split="val")
print(f"Total segments: {len(ds)}")
print(f"Columns: {ds.column_names}")

row = ds[0]
for k, v in row.items():
    print(f"  {k}: {v}")

In [ ]:
# Group segments by video so each video is downloaded only once
videos = defaultdict(list)
for row in ds:
    videos[row["youtube_id"]].append({
        "segment_id" : row["id"],
        "t_start"    : row["segment"][0],
        "t_end"      : row["segment"][1],
        "action_text": row["sentence"],
        "recipe_type": row["recipe_type"],
    })

video_ids = list(videos.keys())
print(f"Unique videos: {len(video_ids)}")
print(f"Avg segments per video: {len(ds) / len(video_ids):.1f}")

# Set PILOT_N=5 for initial validation; increase for full-scale runs
PILOT_N = 5
pilot_ids = video_ids[:PILOT_N]
print(f"Pilot: {PILOT_N} videos -> {sum(len(videos[v]) for v in pilot_ids)} segments")

---
## Stage 3 — Download video segments

Each video is downloaded once at 360p using `yt-dlp`, which is significantly more reliable than `pytube` for fresh downloads. Individual frame pairs are extracted from the downloaded files in Stage 4.

In [ ]:
import subprocess

def download_video(youtube_id: str, output_dir: Path) -> Optional[Path]:
    out_path = output_dir / f"{youtube_id}.mp4"
    if out_path.exists():
        return out_path

    url = f"https://www.youtube.com/watch?v={youtube_id}"
    cmd = [
        "yt-dlp", "-f", "bestvideo[height<=360]+bestaudio/best[height<=360]",
        "--merge-output-format", "mp4",
        "-o", str(out_path), url, "--quiet"
    ]
    result = subprocess.run(cmd, capture_output=True)
    return out_path if out_path.exists() else None


print("Downloading pilot videos...")
downloaded = {}
for vid_id in tqdm(pilot_ids):
    path = download_video(vid_id, VIDEO_DIR)
    if path:
        downloaded[vid_id] = path
    else:
        print(f"  Failed: {vid_id}")

print(f"Downloaded {len(downloaded)}/{len(pilot_ids)} videos")

---
## Stage 4 — Extract frame pairs

For each annotated segment, we extract the frame at `t_start` (G_t) and at `t_end` (G_t+1) using OpenCV. Frames are saved as JPEG for VLM input.

In [ ]:
def extract_frame(video_path: Path, timestamp: float, out_path: Path) -> bool:
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(timestamp * fps))
    ret, frame = cap.read()
    cap.release()
    if not ret:
        return False
    cv2.imwrite(str(out_path), frame)
    return True


frame_pairs = []
for vid_id, vid_path in downloaded.items():
    for seg in videos[vid_id]:
        seg_id = seg["segment_id"]
        f_start = FRAMES_DIR / f"{seg_id}_start.jpg"
        f_end   = FRAMES_DIR / f"{seg_id}_end.jpg"

        ok_s = extract_frame(vid_path, seg["t_start"], f_start)
        ok_e = extract_frame(vid_path, seg["t_end"],   f_end)

        if ok_s and ok_e:
            frame_pairs.append({
                **seg,
                "youtube_id"  : vid_id,
                "frame_start" : str(f_start),
                "frame_end"   : str(f_end),
            })

print(f"Frame pairs extracted: {len(frame_pairs)}")

In [ ]:
# Visual sanity check — inspect the first 3 frame pairs
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

for pair in frame_pairs[:3]:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].imshow(mpimg.imread(pair["frame_start"]))
    axes[0].set_title("Start frame (G_t)")
    axes[0].axis("off")
    axes[1].imshow(mpimg.imread(pair["frame_end"]))
    axes[1].set_title("End frame (G_t+1)")
    axes[1].axis("off")
    fig.suptitle(textwrap.fill(f"Action: {pair['action_text']}", 80), fontsize=10, y=1.02)
    plt.tight_layout()
    plt.show()

---
## Stage 5 — Teacher VLM: scene graph generation

Each frame is sent to Gemini 2.5 Flash Lite with a structured prompt requiring JSON-only output. The schema is intentionally minimal — simpler graphs are easier to validate, train on, and extend later.

**Graph schema:**
```json
{
  "objects": [
    {"id": "obj_1", "class": "knife", "state": {"position": "on_board", "status": "clean"}}
  ],
  "relationships": [
    {"subject": "obj_1", "predicate": "on", "object": "obj_2"}
  ]
}
```

**Rate limiting:** The Gemini Flash free tier allows 15 RPM. Two API calls are made per segment (start + end frame), so a 4.5s sleep is added between segments (~13 RPM). For the full 3k-segment dataset, switch to an async implementation.

In [ ]:
GRAPH_SCHEMA = json.dumps({
    "objects": [
        {"id": "obj_1", "class": "<object_class>", "state": {"<attribute>": "<value>"}}
    ],
    "relationships": [
        {"subject": "obj_1", "predicate": "<spatial_or_action_predicate>", "object": "obj_2"}
    ]
}, indent=2)

SYSTEM_PROMPT = f"""You are a scene graph generator for cooking videos.
Given an image, output a JSON scene graph describing the objects visible and their relationships.

Rules:
- Output ONLY valid JSON. No markdown, no explanation, no backticks.
- List every clearly visible food ingredient, cooking tool, and container.
- Ignore people, hands, and clothing unless they are the primary subject of the action.
- For each object, include relevant state attributes (contents, status, position, quantity).
- Use snake_case for all keys and values.
- Use consistent object IDs: obj_1, obj_2, obj_3 ... ordered by visual prominence.
- Relationships: spatial (on, in, next_to, above, below) and action-based (held_by, poured_into).
- If you are unsure about an object, omit it rather than guessing.

Schema:
{GRAPH_SCHEMA}"""

print("System prompt defined.")

In [ ]:
def call_teacher_vlm(frame_path: str, retries: int = 3) -> Optional[dict]:
    with open(frame_path, "rb") as f:
        img_bytes = f.read()

    image_part = types.Part.from_bytes(data=img_bytes, mime_type="image/jpeg")

    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=TEACHER_MODEL_ID,
                contents=[SYSTEM_PROMPT, image_part],
                config=types.GenerateContentConfig(
                    temperature=0.1,
                    max_output_tokens=2048,
                ),
            )
            raw = response.text.strip()

            # Strip thinking tags (gemini-2.5 models may emit these)
            if "<think>" in raw:
                raw = raw.split("</think>")[-1].strip()

            # Strip markdown code fences if present
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]
                raw = raw.strip()

            graph = json.loads(raw)
            assert "objects"       in graph
            assert "relationships" in graph
            assert isinstance(graph["objects"], list)
            assert isinstance(graph["relationships"], list)
            return graph

        except json.JSONDecodeError as e:
            print(f"  Raw output (first 500 chars):\n{raw[:500]}")
            wait = 2 ** attempt
            print(f"  Attempt {attempt+1} failed (JSONDecodeError). Retrying in {wait}s...")
            time.sleep(wait)
        except Exception as e:
            wait = 2 ** attempt
            print(f"  Attempt {attempt+1} failed ({type(e).__name__}: {e}). Retrying in {wait}s...")
            time.sleep(wait)

    return None


print("Teacher VLM function defined. Model:", TEACHER_MODEL_ID)

In [ ]:
# Single-example test before the full loop
if frame_pairs:
    test_pair = frame_pairs[0]
    print(f"Segment : {test_pair['segment_id']}")
    print(f"Action  : {test_pair['action_text']}\n")

    g_t = call_teacher_vlm(test_pair["frame_start"])
    print("G_t (start frame):")
    print(json.dumps(g_t, indent=2))

    time.sleep(4)

    g_t1 = call_teacher_vlm(test_pair["frame_end"])
    print("\nG_t+1 (end frame):")
    print(json.dumps(g_t1, indent=2))

In [ ]:
# Main annotation loop
# Results are cached to per-segment JSON files so progress survives runtime disconnects.

API_SLEEP_S = 4.5  # ~13 RPM — safely below the 15 RPM free-tier limit

raw_results = []

for pair in tqdm(frame_pairs, desc="Segments"):
    seg_id     = pair["segment_id"]
    cache_path = GRAPHS_DIR / f"{seg_id}.json"

    if cache_path.exists():
        with open(cache_path) as f:
            raw_results.append(json.load(f))
        continue

    g_t = call_teacher_vlm(pair["frame_start"])
    time.sleep(API_SLEEP_S)
    g_t1 = call_teacher_vlm(pair["frame_end"])
    time.sleep(API_SLEEP_S)

    if g_t is None or g_t1 is None:
        tqdm.write(f"  Skipping {seg_id}: VLM call failed")
        continue

    result = {
        "segment_id"  : seg_id,
        "youtube_id"  : pair["youtube_id"],
        "recipe_type" : pair["recipe_type"],
        "t_start"     : pair["t_start"],
        "t_end"       : pair["t_end"],
        "action_text" : pair["action_text"],
        "g_t"         : g_t,
        "g_t1"        : g_t1,
        "frame_start" : pair["frame_start"],
        "frame_end"   : pair["frame_end"],
    }

    with open(cache_path, "w") as f:
        json.dump(result, f)

    raw_results.append(result)

print(f"Raw results: {len(raw_results)} segments processed")

---
## Stage 6 — Consistency filtering

Three checks are applied to each raw result. Triplets that fail any check are discarded.

1. **Minimum objects** — both G_t and G_t+1 must contain at least one object. Empty graphs indicate a black frame, extreme blur, or VLM failure.

2. **Semantic object overlap** — at least 50% of object *classes* in G_t must have a semantically similar match in G_t+1 (cosine similarity >= 0.5 on `all-MiniLM-L6-v2` embeddings). This catches cases where the VLM hallucinated an entirely different scene.

3. **State change exists** — G_t and G_t+1 must not be byte-for-byte identical. A triplet where nothing changed provides no training signal.

Thresholds can be adjusted based on the failure breakdown printed below.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

MIN_OBJECTS          = 1
SIMILARITY_THRESHOLD = 0.5


def get_object_classes(graph: dict) -> set:
    return {obj.get("class", "") for obj in graph.get("objects", [])}


def embed_classes(classes: set) -> np.ndarray:
    if not classes:
        return np.zeros((0, 384))
    return embedder.encode(list(classes), normalize_embeddings=True)


def semantic_overlap(classes_t: set, classes_t1: set) -> float:
    """
    For each class in G_t, find the most similar class in G_t+1.
    Returns the fraction of G_t classes that exceed the similarity threshold.
    Directional: measures how much of G_t is covered by G_t+1.
    """
    if not classes_t or not classes_t1:
        return 0.0
    emb_t  = embed_classes(classes_t)
    emb_t1 = embed_classes(classes_t1)
    sim_matrix  = emb_t @ emb_t1.T
    best_scores = sim_matrix.max(axis=1)
    matched = (best_scores >= SIMILARITY_THRESHOLD).sum()
    return matched / len(classes_t)


def is_consistent(result: dict) -> tuple:
    g_t  = result["g_t"]
    g_t1 = result["g_t1"]

    if len(g_t.get("objects", []))  < MIN_OBJECTS:
        return False, "g_t has too few objects"
    if len(g_t1.get("objects", [])) < MIN_OBJECTS:
        return False, "g_t1 has too few objects"

    classes_t  = get_object_classes(g_t)
    classes_t1 = get_object_classes(g_t1)
    overlap = semantic_overlap(classes_t, classes_t1)
    if overlap < 0.5:
        return False, f"low semantic overlap ({overlap:.0%}): {classes_t} vs {classes_t1}"

    if json.dumps(g_t, sort_keys=True) == json.dumps(g_t1, sort_keys=True):
        return False, "g_t and g_t1 are identical (no state change)"

    return True, "ok"


triplets    = []
fail_reasons = []

for result in raw_results:
    passes, reason = is_consistent(result)
    if passes:
        triplets.append(result)
    else:
        fail_reasons.append(reason)

print(f"Raw:          {len(raw_results)}")
print(f"After filter: {len(triplets)}  ({len(triplets)/max(len(raw_results),1):.0%} pass rate)")
print(f"Dropped:      {len(fail_reasons)}")

if fail_reasons:
    from collections import Counter
    print("\nFailure breakdown:")
    for reason, count in Counter(fail_reasons).most_common():
        print(f"  {count:3d}x  {reason}")

In [ ]:
# Qualitative inspection of passing triplets
for triplet in triplets[:2]:
    print("=" * 60)
    print(f"Segment  : {triplet['segment_id']}")
    print(f"Recipe   : {triplet['recipe_type']}")
    print(f"Action   : {triplet['action_text']}")
    print("\nG_t:")
    print(json.dumps(triplet["g_t"], indent=2))
    print("\nG_t+1:")
    print(json.dumps(triplet["g_t1"], indent=2))
    print()

---
## Stage 7 — Save final triplet dataset

Output format is **JSONL** (one JSON object per line). Advantages:
- Streamable in a PyTorch `DataLoader` without loading the full dataset into memory
- Inspectable with standard CLI tools (`head`, `jq`, pandas)
- Compatible with `datasets.load_dataset("json", ...)`

A lightweight manifest (no graph data) is also saved for quick statistics.

In [ ]:
with open(OUTPUT_FILE, "w") as f:
    for triplet in triplets:
        f.write(json.dumps(triplet) + "\n")

size_mb = OUTPUT_FILE.stat().st_size / 1e6
print(f"Saved {len(triplets)} triplets -> {OUTPUT_FILE} ({size_mb:.2f} MB)")

manifest = {
    "total_triplets"  : len(triplets),
    "source_videos"   : len(set(t["youtube_id"] for t in triplets)),
    "split"           : "val",
    "filter_pass_rate": f"{len(triplets)/max(len(raw_results),1):.2%}",
    "teacher_model"   : TEACHER_MODEL_ID,
}
manifest_path = ROOT / "manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("\nManifest:")
print(json.dumps(manifest, indent=2))
print(f"\nPhase 1 complete. Upload triplets.jsonl to Phase 2.")